# AP02 – Data Ingestion (V4 Part 1)
## 1. Web Scraping mit BeautifulSoup

#### Die Unterschiede zwischen den vier Beschaffungsmethoden solltest du sicher beherrschen: BeautifulSoup analysiert HTML-Code, Selenium steuert einen Browser und kann mit interaktiven Webseiten arbeiten, REST APIs liefern strukturierte Daten über HTTP, RSS Feeds liefern standardisierte Informationen über neue Veröffentlichungen.


- **Definition:** Web Scraping bezeichnet das automatisierte Auslesen und Extrahieren von Informationen aus Webseiten. BeautifulSoup ist eine Python-Bibliothek, die HTML- und XML-Dokumente analysiert und deren Inhalte gezielt zugänglich macht.
- **Zweck:** Informationen wie Produktnamen, Preise oder Verfügbarkeiten aus HTML extrahieren und für die weitere Datenanalyse in einem DataFrame speichern.
- **Funktionsweise:**
  1. HTML-Code aus einer lokalen Datei oder von einer Webseite einlesen.
  2. HTML-Code mit `BeautifulSoup()` parsen.
  3. Gewünschte HTML-Elemente anhand ihrer Tags oder CSS-Klassen suchen.
  4. Texte und Attribute aus den gefundenen Elementen extrahieren.
  5. Extrahierte Informationen mit `pd.DataFrame()` tabellarisch organisieren.

### Wichtige Methoden

- **`open('datei.html', encoding='utf-8')`:** Lokale HTML-Datei öffnen.
- **`.read()`:** Vollständigen Dateiinhalt als Text einlesen.
- **`BeautifulSoup(html_content, 'html.parser')`:** HTML-Code analysieren und als durchsuchbare Baumstruktur darstellen.
- **`.find('tag', class_='klasse')`:** Erstes passendes HTML-Element finden.
- **`.find_all('tag', class_='klasse')`:** Alle passenden HTML-Elemente als Liste zurückgeben.
- **`.select('.klasse')`:** Alle Elemente mit einem CSS-Selektor auswählen.
- **`.get_text(strip=True)`:** Textinhalt eines Elements extrahieren und äusseren Leerraum entfernen.
- **`.get('attribut')`:** Wert eines HTML-Attributs auslesen, beispielsweise `data-product-id`.
- **`pd.DataFrame(liste)`:** Extrahierte Datensätze in einen pandas-DataFrame umwandeln.
- **`df.to_csv('datei.csv', index=False)`:** Extrahierte Daten als CSV-Datei speichern.

### HTML-Grundlagen

- **Tag:** HTML-Element wie `<div>`, `<span>`, `<h2>` oder `<table>`.
- **Class:** CSS-Klasse eines Elements, beispielsweise `class="product-card"`.
- **Attribute:** Zusätzliche Informationen innerhalb eines Tags, beispielsweise `data-product-id="PM-10001"`.
- **DOM (Document Object Model):** Hierarchische Baumstruktur eines HTML-Dokuments.

### Erkenntnisse

- **`find()` vs. `find_all()`:** `find()` liefert das erste passende Element; `find_all()` liefert alle passenden Elemente.
- **BeautifulSoup vs. Selenium:** BeautifulSoup analysiert vorhandenen HTML-Code; Selenium steuert einen echten Browser und kann auch mit dynamischen und interaktiven Webseiten arbeiten.
- **Web Scraping vs. Web Crawling:** Scraping extrahiert konkrete Inhalte; Crawling entdeckt und verfolgt Webseiten beziehungsweise Links.
- **Achtung:** HTML-Strukturen unterscheiden sich je nach Webseite. Werden Klassen oder Tags verändert, müssen die Suchausdrücke entsprechend angepasst werden.
- **Voraussetzung:** Die HTML-Datei muss vorhanden sein und die gesuchten Elemente enthalten.

#### Wichtig: Der erzeugte DataFrame heisst bewusst df_scraped. Damit überschreiben wir weder df_autoscout noch df_apartments oder df_smartphone. Die Produktpreise bleiben zunächst Rohtexte; ihre numerische Bereinigung gehört zur Data Preparation.

In [2]:
# Import libraries
from bs4 import BeautifulSoup                                                   #BeautifulSoup importieren: HTML-Dokumente analysieren und Elemente auslesen
import pandas as pd                                                             #pandas importieren: Extrahierte Daten als DataFrame speichern

# Read HTML from a local file
with open('./html/shop_page_1.html', 'r', encoding='utf-8') as file:            #HTML-Datei aus dem Unterordner html öffnen
    html_content = file.read()                                                  #read(): Vollständigen HTML-Code als Text einlesen

# Parse the HTML content
soup = BeautifulSoup(html_content, 'html.parser')                               #HTML-Code parsen und durchsuchbare Baumstruktur erstellen
                                                                                #'html.parser': In Python integrierten HTML-Parser verwenden

# Find all product cards
products = soup.find_all('div', class_='product-card')                          #Alle div-Elemente mit der CSS-Klasse product-card suchen
                                                                                #find_all(): Liste sämtlicher passender HTML-Elemente zurückgeben

# Extract product information
rows = []                                                                       #Leere Liste erstellen, um die extrahierten Produktinformationen zu sammeln

for product in products:                                                        #Alle gefundenen Produktkarten einzeln durchlaufen

    product_id = product.get('data-product-id')                                 #Produkt-ID aus dem HTML-Attribut data-product-id auslesen

    brand = product.find('span', class_='brand').get_text(strip=True)           #Erstes span-Element mit Klasse brand suchen und Text extrahieren
                                                                                #get_text(strip=True): HTML entfernen und äusseren Leerraum entfernen

    model = product.find('span', class_='model').get_text(strip=True)           #Produktmodell aus dem entsprechenden HTML-Element extrahieren

    price = product.find('span', class_='price').get_text(strip=True)           #Aktuellen Produktpreis als Text extrahieren
                                                                                #Ergebnis beispielsweise 'CHF 299.–'

    availability = product.find('span', class_='availability').get_text(strip=True) #Verfügbarkeit des Produkts auslesen

    rows.append({                                                               #Extrahierte Informationen als Dictionary zur Liste hinzufügen
        'product_id': product_id,                                               #Eindeutige Produkt-ID speichern
        'brand': brand,                                                         #Marke speichern
        'model': model,                                                         #Modell speichern
        'price_raw': price,                                                     #Preis zunächst als Rohtext speichern
        'availability': availability                                            #Verfügbarkeit speichern
    })

# Create a DataFrame
df_scraped = pd.DataFrame(rows)                                                 #Liste der Dictionaries in einen pandas-DataFrame umwandeln
                                                                                #Jedes Dictionary ergibt eine Zeile; die Dictionary-Schlüssel werden zu Spalten

# Show extracted data
print('Number of products:', len(df_scraped))                                   #Anzahl extrahierter Produkte anzeigen
df_scraped.head()                                                               #Erste fünf Produktdatensätze anzeigen

# Save extracted data
df_scraped.to_csv('smartphones_scraped.csv', index=False)                       #Extrahierte Daten als CSV-Datei speichern
                                                                                #index=False: Automatischen DataFrame-Index nicht mitspeichern

#ANPASSEN: './html/shop_page_1.html' an den tatsächlichen Pfad deiner HTML-Datei anpassen
#ANPASSEN: 'div' und 'product-card' an die HTML-Elemente und CSS-Klassen der Zielwebseite anpassen
#ANPASSEN: 'brand', 'model', 'price' und 'availability' an die vorhandenen CSS-Klassen anpassen
#ANPASSEN: 'data-product-id' an das gewünschte HTML-Attribut anpassen
#ANPASSEN: Dictionary-Schlüssel an die gewünschten Namen der Ergebnisspalten anpassen
#ANPASSEN: 'smartphones_scraped.csv' an den gewünschten Ausgabedateinamen anpassen
#NICHTS ANPASSEN: BeautifulSoup(), find_all(), find(), get_text(), pd.DataFrame() und to_csv() als Methoden
#VORAUSSETZUNG: pandas und beautifulsoup4 installiert; HTML-Datei liegt am angegebenen Speicherort
#ACHTUNG: Bei fehlenden HTML-Elementen gibt find() None zurück; anschliessendes get_text() erzeugt dann einen Fehler
#ERKENNTNIS: Die HTML-Struktur wird in eine Tabelle umgewandelt, deren Daten anschliessend bereinigt und analysiert werden können

Number of products: 47


## 2. Web Scraping mit Selenium & XPath

- **Definition:** Selenium ist ein Framework zur automatisierten Steuerung von Webbrowsern. Damit können Webseiten geöffnet, HTML-Elemente gefunden und Benutzerinteraktionen automatisiert werden.
- **Zweck:** Daten von Webseiten extrahieren, insbesondere wenn Inhalte dynamisch mit JavaScript geladen werden oder Benutzerinteraktionen erforderlich sind.
- **WebDriver:** Schnittstelle, über die Python den Webbrowser steuert.
- **Headless Mode:** Browser ohne sichtbare grafische Oberfläche ausführen, beispielsweise auf einem Server oder in GitHub Codespaces.

### Wichtige Methoden

- **`webdriver.Chrome()`:** Chrome-Browser über Selenium starten.
- **`webdriver.ChromeOptions()`:** Browser-Einstellungen definieren.
- **`options.add_argument('--headless=new')`:** Chrome ohne sichtbares Browserfenster starten.
- **`driver.get(url)`:** Webseite beziehungsweise lokale HTML-Datei öffnen.
- **`driver.title`:** Titel der aktuellen Webseite auslesen.
- **`driver.current_url`:** Aktuelle Webseitenadresse ausgeben.
- **`driver.page_source`:** Vollständigen aktuellen HTML-Code der Webseite abrufen.
- **`driver.find_element()`:** Erstes passendes HTML-Element finden.
- **`driver.find_elements()`:** Alle passenden Elemente als Liste zurückgeben.
- **`element.text`:** Sichtbaren Textinhalt eines HTML-Elements auslesen.
- **`element.get_attribute('name')`:** Wert eines HTML-Attributs auslesen.
- **`element.click()`:** HTML-Element anklicken.
- **`element.send_keys('Text')`:** Text in ein Eingabefeld schreiben.
- **`driver.save_screenshot('bild.png')`:** Screenshot der Webseite speichern.
- **`driver.quit()`:** Browser und WebDriver-Sitzung vollständig beenden.

### HTML-Elemente lokalisieren

- **`By.ID`:** Element anhand einer eindeutigen ID suchen.
- **`By.NAME`:** Element anhand des HTML-Attributs `name` suchen.
- **`By.CLASS_NAME`:** Element anhand seiner CSS-Klasse suchen.
- **`By.TAG_NAME`:** Element anhand seines HTML-Tags suchen, beispielsweise `h1`.
- **`By.CSS_SELECTOR`:** Element mit einem CSS-Selektor auswählen, beispielsweise `.product-card`.
- **`By.XPATH`:** Element über einen XPath-Ausdruck im HTML-Baum finden.
- **`By.LINK_TEXT`:** Link anhand seines vollständigen sichtbaren Textes finden.

### XPath

- **Definition:** XPath ist eine Sprache zur Auswahl bestimmter Elemente innerhalb einer XML- oder HTML-Baumstruktur.
- **`//div`:** Alle `div`-Elemente im Dokument auswählen.
- **`//div[@class='product-card']`:** `div`-Elemente mit der angegebenen CSS-Klasse auswählen.
- **`//span[@class='price']`:** Preis-Elemente anhand ihrer CSS-Klasse auswählen.
- **`.//span[@class='model']`:** Innerhalb eines bereits gefundenen Elements nach einem passenden `span` suchen.
- **`//div[@id='content']`:** Element mit der ID `content` auswählen.
- **`//a[contains(@href, 'produkt')]`:** Links auswählen, deren `href`-Attribut den Text `produkt` enthält.
- **Wichtig:** `//` sucht im Dokument nach passenden Elementen; `.//` sucht relativ zum aktuell ausgewählten Element.

### WebDriverWait

- **`WebDriverWait(driver, 10)`:** Maximal zehn Sekunden auf eine Bedingung warten.
- **`EC.presence_of_element_located()`:** Warten, bis ein HTML-Element im DOM vorhanden ist.
- **`EC.element_to_be_clickable()`:** Warten, bis ein Element sichtbar und anklickbar ist.
- **Zweck:** Verhindern, dass Selenium Elemente sucht, bevor eine Webseite sie bereitgestellt hat.

### Erkenntnisse

- **BeautifulSoup:** Analysiert vorhandenen HTML-Code, steuert aber keinen Browser.
- **Selenium:** Steuert einen echten Browser und kann mit dynamischen Webseiten interagieren.
- **find_element() vs. find_elements():** Erstes Element beziehungsweise Liste aller passenden Elemente.
- **XPath vs. CSS Selector:** Beide identifizieren HTML-Elemente, verwenden aber unterschiedliche Ausdrücke.
- **Achtung:** Selenium benötigt einen Browser und einen passenden WebDriver. Die Ausführung ist deshalb aufwendiger als BeautifulSoup.

In [ ]:
#Falls Selenium nichtl läuft => %pip install selenium

import selenium

print('Selenium version:', selenium.__version__) #Installierte Selenium-Version anzeigen


import site

site.addsitedir('/home/vscode/.local/lib/python3.12/site-packages') #Installierten Paketordner zum Python-Suchpfad hinzufügen

import selenium

print('Selenium version:', selenium.__version__) #Überprüfen, ob Selenium jetzt importiert werden kann

Selenium version: 4.51.0


In [8]:
#Falls er nicht läuft %pip install selenium
# Import libraries
from pathlib import Path                                                        #Path: Dateipfade verwalten und in lokale URLs umwandeln
from selenium import webdriver                                                  #Selenium WebDriver zur Browserautomatisierung importieren
from selenium.webdriver.common.by import By                                     #By: Suchstrategie für HTML-Elemente festlegen
from selenium.webdriver.support.ui import WebDriverWait                         #WebDriverWait: Auf HTML-Elemente warten
from selenium.webdriver.support import expected_conditions as EC                #EC: Bedingungen definieren, die erfüllt sein müssen
import pandas as pd                                                             #pandas: Extrahierte Informationen als DataFrame speichern

# Set up Chrome options
options = webdriver.ChromeOptions()                                             #Chrome-Browseroptionen erstellen
options.add_argument('--headless=new')                                          #Browser ohne sichtbare grafische Oberfläche starten

# Define local HTML file
html_path = Path('./html/shop_page_1.html').resolve()                           #Absoluten Pfad zur lokalen HTML-Datei bestimmen
                                                                                #resolve(): Relativen Dateipfad in absoluten Pfad umwandeln

# Initialize the driver
driver = webdriver.Chrome(options=options)                                      #Chrome-Browser mit den definierten Einstellungen starten
                                                                                #Voraussetzung: Chrome/Chromium und passender WebDriver verfügbar

try:
    # Open the HTML file
    driver.get(html_path.as_uri())                                              #Lokale HTML-Datei im Browser öffnen
                                                                                #as_uri(): Dateipfad in eine file://-URL umwandeln

    # Show browser information
    print('Page title:', driver.title)                                          #Titel der geöffneten Webseite anzeigen
    print('Current URL:', driver.current_url)                                   #Aktuelle Datei- bzw. Webseitenadresse anzeigen

    # Wait until product cards are available
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located(
            (By.CSS_SELECTOR, '.product-card')
        )
    )                                                                           #Maximal zehn Sekunden warten, bis mindestens eine Produktkarte im DOM vorhanden ist
                                                                                #By.CSS_SELECTOR: HTML-Element anhand seiner CSS-Klasse suchen
                                                                                #EC.presence_of_element_located(): Prüfen, ob das Element im DOM existiert

    # Locate all product cards
    products = driver.find_elements(By.CLASS_NAME, 'product-card')              #Alle Elemente mit der Klasse product-card suchen
                                                                                #find_elements(): Liste passender WebElement-Objekte zurückgeben

    # Extract product information
    rows = []                                                                   #Leere Liste zum Sammeln der extrahierten Produktdaten erstellen

    for product in products:                                                    #Alle gefundenen Produktkarten einzeln durchlaufen

        product_id = product.get_attribute('data-product-id')                   #Produkt-ID aus einem HTML-Attribut auslesen

        brand = product.find_element(By.CSS_SELECTOR, '.brand').text            #Markenname über einen CSS-Selektor finden
                                                                                #.text: Sichtbaren Textinhalt des Elements auslesen

        model = product.find_element(By.XPATH, './/span[@class="model"]').text  #Modell mit einem relativen XPath-Ausdruck suchen
                                                                                #.//: Innerhalb der aktuellen Produktkarte suchen
                                                                                #@class="model": Element mit passendem class-Attribut

        price = product.find_element(By.CSS_SELECTOR, '.price').text            #Produktpreis anhand seiner CSS-Klasse extrahieren

        rows.append({                                                           #Informationen als Dictionary in einer Liste speichern
            'product_id': product_id,                                           #Produkt-ID speichern
            'brand': brand,                                                     #Marke speichern
            'model': model,                                                     #Modell speichern
            'price_raw': price                                                  #Preis als Text speichern
        })

    # Create a DataFrame
    df_selenium = pd.DataFrame(rows)                                            #Extrahierte Produktinformationen in einen DataFrame umwandeln

    # Show extracted data
    print('Number of products:', len(df_selenium))                              #Anzahl extrahierter Produktkarten anzeigen
    display(df_selenium.head())                                                 #Erste fünf Produkte anzeigen

    # Save a screenshot
    driver.save_screenshot('shop_selenium.png')                                 #Screenshot des aktuellen Browserfensters als PNG speichern

finally:
    # Close the browser
    driver.quit()                                                               #Browser und WebDriver-Sitzung auch bei einem Fehler vollständig beenden

#ANPASSEN: './html/shop_page_1.html' durch die gewünschte HTML-Datei ersetzen
#ANPASSEN: Für eine Online-Webseite driver.get('https://...') statt html_path.as_uri() verwenden
#ANPASSEN: 'product-card', '.brand', '.price' und den XPath-Ausdruck an die HTML-Struktur anpassen
#ANPASSEN: 'data-product-id' an das gewünschte HTML-Attribut anpassen
#ANPASSEN: Dictionary-Schlüssel an die gewünschten Ergebnisspalten anpassen
#OPTIONAL: '--headless=new' entfernen, wenn der Browser sichtbar geöffnet werden soll
#OPTIONAL: WebDriverWait(driver, 10) an die gewünschte maximale Wartezeit anpassen
#NICHTS ANPASSEN: webdriver.Chrome(), By.CSS_SELECTOR, By.XPATH, find_elements(), find_element() und driver.quit() als Funktionen beziehungsweise Methoden
#VORAUSSETZUNG: selenium und pandas installiert; Chrome/Chromium und ein kompatibler ChromeDriver verfügbar
#VORAUSSETZUNG: Die lokale HTML-Datei muss existieren
#ACHTUNG: Ohne installierten Browser kann die WebDriver-Initialisierung fehlschlagen, auch wenn selenium installiert ist
#ERKENNTNIS: Selenium automatisiert die Browsersteuerung; BeautifulSoup analysiert HTML ohne Browser

WebDriverException: Message: Service /home/vscode/.cache/selenium/chromedriver/linux64/155.0.8059.84/chromedriver unexpectedly exited. Status code was: 127


## 3. Data Ingestion: REST APIs mit Requests & JSON

- **Definition:** Eine API (Application Programming Interface) ist eine Schnittstelle zum Austausch von Daten und Funktionen zwischen Anwendungen. Eine REST API stellt Ressourcen über HTTP zur Verfügung.
- **Zweck:** Strukturierte Daten direkt von einem Webdienst abrufen, ohne die HTML-Struktur einer Webseite analysieren zu müssen.
- **Funktionsweise:**
  1. API-Endpunkt (Endpoint) festlegen.
  2. HTTP-Anfrage mit der passenden Methode senden.
  3. HTTP-Antwort und Statuscode überprüfen.
  4. Antwortdaten, beispielsweise JSON, auslesen.
  5. JSON-Daten in einen pandas-DataFrame umwandeln.

### HTTP-Methoden

- **GET:** Daten von einem Server abrufen, beispielsweise aktuelle Kryptowährungspreise.
- **POST:** Daten an einen Server senden, beispielsweise eine komplexe Suchanfrage an die Overpass API.
- **PUT:** Ressource vollständig ersetzen beziehungsweise erstellen.
- **PATCH:** Teile einer Ressource aktualisieren.
- **DELETE:** Ressource löschen.

### HTTP Request und Response

- **Request:** Anfrage eines Clients an einen Server.
- **Endpoint:** URL einer bestimmten API-Ressource oder Funktion.
- **Parameters:** Zusätzliche Angaben zur Anfrage, beispielsweise Suchbegriffe oder Filter.
- **Headers:** Metadaten wie Authentifizierung oder gewünschtes Datenformat.
- **Body:** Optionaler Inhalt, der mit der Anfrage übertragen wird.
- **Response:** Antwort des Servers mit Statuscode, Metadaten und gegebenenfalls Daten.
- **HTTP 200:** Anfrage erfolgreich.
- **HTTP 400:** Ungültige Anfrage.
- **HTTP 401:** Authentifizierung erforderlich beziehungsweise fehlgeschlagen.
- **HTTP 403:** Zugriff verweigert.
- **HTTP 404:** Ressource nicht gefunden.
- **HTTP 429:** Zu viele Anfragen (Rate Limit).
- **HTTP 500:** Interner Serverfehler.

### Wichtige Python-Methoden

- **`requests.get(url, params=...)`:** GET-Anfrage mit optionalen URL-Parametern senden.
- **`requests.post(url, data=...)`:** POST-Anfrage mit Daten im Request Body senden.
- **`timeout=30`:** Anfrage nach maximal 30 Sekunden Wartezeit abbrechen.
- **`response.status_code`:** HTTP-Statuscode der Antwort anzeigen.
- **`response.raise_for_status()`:** Bei einem HTTP-Fehlerstatus eine Exception auslösen.
- **`response.json()`:** JSON-Antwort in Python-Objekte wie Dictionaries oder Listen umwandeln.
- **`json.load()`:** JSON-Daten aus einer geöffneten Datei einlesen.
- **`json.dump()`:** Python-Daten als JSON in eine Datei schreiben.
- **`pd.json_normalize()`:** Verschachtelte JSON-Daten in eine flache tabellarische Struktur umwandeln.
- **`.reindex(columns=[...])`:** Gewünschte Spalten auswählen und in einer bestimmten Reihenfolge anordnen.

### JSON-Verarbeitung

- **JSON Object `{}`:** Sammlung von Schlüssel-Wert-Paaren, in Python normalerweise als `dict` dargestellt.
- **JSON Array `[]`:** Sammlung von Elementen, in Python normalerweise als `list` dargestellt.
- **Nested JSON:** Verschachtelte Objekte, beispielsweise geografische Koordinaten innerhalb eines `center`-Objekts.
- **`pd.json_normalize()`:** Macht verschachtelte Werte als DataFrame-Spalten zugänglich, beispielsweise `center.lat`.

### Beispiele aus der Vorlesung

- **CoinGecko API:** Preise, Marktkapitalisierungen und weitere Informationen zu Kryptowährungen abrufen.
- **Flickr API:** Informationen über Fotos abrufen.
- **Overpass API:** Geografische Informationen aus OpenStreetMap gezielt abfragen.

### Erkenntnisse

- REST APIs liefern Daten in einem definierten Format, häufig JSON.
- GET eignet sich zum Abrufen von Informationen; POST kann beispielsweise komplexe Suchanfragen übertragen.
- `.json()` wandelt eine API-Antwort in Python-Datenstrukturen um.
- `pd.json_normalize()` wandelt verschachtelte JSON-Objekte in einen analysierbaren DataFrame um.
- API-Anfragen können aufgrund fehlender Internetverbindung, Authentifizierung, Rate Limits oder Serverproblemen fehlschlagen.
- **Unterschied zu Web Scraping:** APIs liefern Daten über eine definierte Schnittstelle; Scraping extrahiert Informationen aus Webseiten.

In [5]:
#Variante 1: GET mit CoinGecko
# Import libraries
import requests                                                                 #HTTP-Anfragen an Web APIs senden
import pandas as pd                                                             #API-Daten als DataFrame verarbeiten

# Define API endpoint
url = 'https://api.coingecko.com/api/v3/coins/markets'                          #API-Endpunkt für Kryptowährungsmarktdaten

# Define request parameters
params = {                                                                      #Parameter für die API-Abfrage festlegen
    'vs_currency': 'chf',                                                       #Preise in Schweizer Franken abrufen
    'ids': 'bitcoin,ethereum'                                                   #Informationen zu Bitcoin und Ethereum abrufen
}

# Send GET request
response = requests.get(url, params=params, timeout=30)                         #GET-Anfrage mit Parametern an CoinGecko senden
                                                                                #params: Parameter automatisch an die URL anhängen
                                                                                #timeout=30: Wartezeit auf maximal 30 Sekunden begrenzen

# Check HTTP response
print('Status code:', response.status_code)                                     #HTTP-Statuscode der Antwort anzeigen
response.raise_for_status()                                                     #Bei HTTP-Fehlerstatus eine Exception auslösen

# Convert JSON response to Python data
data = response.json()                                                          #JSON-Antwort in Python-Objekte (hier eine Liste von Dictionaries) umwandeln

# Create a DataFrame
df_api_coins = pd.json_normalize(data)                                          #JSON-Daten in einen pandas-DataFrame umwandeln

# Show relevant columns
df_api_coins[['id', 'symbol', 'current_price', 'market_cap']].head()            #Kryptowährung, Symbol, Preis und Marktkapitalisierung anzeigen

#ANPASSEN: url an den gewünschten API-Endpunkt anpassen
#ANPASSEN: params an die von der API unterstützten Parameter anpassen
#ANPASSEN: ['id', 'symbol', 'current_price', 'market_cap'] an die verfügbaren JSON-Felder anpassen
#NICHTS ANPASSEN: requests.get(), raise_for_status(), response.json() und pd.json_normalize()
#VORAUSSETZUNG: Internetzugang; requests und pandas installiert
#ACHTUNG: Die API kann aufgrund von Rate Limits oder Zugriffsregeln einen Fehler zurückgeben
#ERKENNTNIS: GET ruft Daten direkt über eine strukturierte Schnittstelle ab

Status code: 200


,id,symbol,current_price,market_cap
0,bitcoin,btc,67679.00,1360234714029
1,ethereum,eth,1993.19,243446066869


In [6]:
#Variante 2: POST mit Overpass API
# Import libraries
import requests                                                                 #HTTP-Anfragen senden
import pandas as pd                                                             #JSON-Daten tabellarisch verarbeiten

# Define API endpoint
OVERPASS_URL = 'https://overpass.osm.ch/api/interpreter'                        #Overpass-API-Endpunkt für OpenStreetMap-Daten

# Define Overpass query
query = """
[out:json][timeout:90];
area["name"="Zürich"]["admin_level"="8"]->.searchArea;
nwr["amenity"="drinking_water"](area.searchArea);
out center;
"""                                                                             #Overpass-Abfrage für Trinkwasserstellen in der Stadt Zürich
                                                                                #[out:json]: Antwortformat JSON
                                                                                #area: Untersuchungsgebiet festlegen
                                                                                #nwr: Nodes, Ways und Relations mit amenity=drinking_water suchen
                                                                                #out center: Bei Flächen und Linien zusätzlich Mittelpunkt-Koordinaten zurückgeben

# Send POST request
response = requests.post(
    OVERPASS_URL,                                                               #API-Endpunkt
    data={'data': query},                                                       #Overpass-Abfrage im Request Body übertragen
    timeout=90)                                                                 #Maximal 90 Sekunden auf die Antwort warten

# Check HTTP response
print('Status code:', response.status_code)                                     #HTTP-Statuscode anzeigen
response.raise_for_status()                                                     #Bei HTTP-Fehlerstatus Exception auslösen

# Extract JSON data
data = response.json()                                                          #JSON-Antwort in Python-Datenstrukturen umwandeln
elements = data['elements']                                                     #Liste der gefundenen OpenStreetMap-Objekte aus dem Dictionary auslesen

# Normalize nested JSON
df_api_raw = pd.json_normalize(elements)                                        #Verschachtelte JSON-Objekte in DataFrame-Spalten umwandeln
                                                                                #Beispiel: center.lat wird zu einem eigenen Spaltennamen

# Select relevant columns
df_api = df_api_raw.reindex(
    columns=['id', 'type', 'lat', 'lon', 'center.lat', 'center.lon', 'tags.name']
) #Benötigte Felder auswählen; fehlende Felder werden als NaN ergänzt

# Complete geographical coordinates
df_api['lat'] = df_api['lat'].fillna(df_api['center.lat'])                      #Fehlenden Breitengrad durch Mittelpunkt-Breitengrad ersetzen
df_api['lon'] = df_api['lon'].fillna(df_api['center.lon'])                      #Fehlenden Längengrad durch Mittelpunkt-Längengrad ersetzen

# Rename and select final columns
df_api = df_api[['id', 'type', 'lat', 'lon', 'tags.name']].rename(
    columns={'id': 'osm_id', 'type': 'osm_type', 'tags.name': 'name'}
)                                                                               #Spalten auswählen und verständliche Namen vergeben

# Show data
print('Number of elements:', len(df_api))                                       #Anzahl gefundener Trinkwasserstellen anzeigen
df_api.head()                                                                   #Erste fünf Datensätze anzeigen

# Save data
df_api.to_csv('drinking_fountains.csv', index=False)                            #API-Ergebnis als CSV-Datei speichern

#ANPASSEN: OVERPASS_URL an den gewünschten API-Endpunkt anpassen
#ANPASSEN: query an das gesuchte Gebiet und die gewünschten OpenStreetMap-Merkmale anpassen
#ANPASSEN: columns=[...] an die tatsächlich gelieferten JSON-Felder anpassen
#ANPASSEN: 'drinking_fountains.csv' an den gewünschten Ausgabedateinamen anpassen
#NICHTS ANPASSEN: requests.post(), response.json(), pd.json_normalize(), .fillna() und .to_csv() als Methoden
#VORAUSSETZUNG: Internetzugang; requests und pandas installiert
#ACHTUNG: Der Overpass-Server kann überlastet sein; bei HTTP-Fehlern Anfrage später erneut versuchen
#ERKENNTNIS: Verschachtelte JSON-Daten lassen sich mit json_normalize() in einen tabellarischen DataFrame umwandeln

Status code: 200
Number of elements: 951


## 4. Data Ingestion: RSS Feeds mit feedparser

- **Definition:** RSS (Really Simple Syndication) ist ein standardisiertes, XML-basiertes Webfeed-Format zur automatischen Bereitstellung neuer Online-Inhalte.
- **Zweck:** Nachrichten, Blogbeiträge und andere regelmässig veröffentlichte Inhalte automatisch abrufen und auswerten.
- **Funktionsweise:**
  1. RSS-Feed über eine URL abrufen.
  2. XML-Inhalte mit einem RSS-Parser interpretieren.
  3. Einzelne Beiträge und deren Metadaten extrahieren.
  4. Titel, Links, Veröffentlichungsdaten und Beschreibungen tabellarisch speichern.

### RSS-Struktur

- **`<rss>`:** Wurzelelement des RSS-Dokuments.
- **`<channel>`:** Allgemeine Informationen über den RSS-Feed.
- **`<title>`:** Titel des Feeds beziehungsweise eines Artikels.
- **`<link>`:** URL des Feeds beziehungsweise eines Artikels.
- **`<description>`:** Beschreibung oder Zusammenfassung.
- **`<item>`:** Einzelner Beitrag innerhalb des Feeds.
- **`<pubDate>`:** Veröffentlichungsdatum eines Beitrags.
- **`<guid>`:** Eindeutige Kennung eines Beitrags.

### Wichtige Python-Methoden

- **`import feedparser`:** Bibliothek zum Einlesen und Interpretieren von RSS-Feeds importieren.
- **`feedparser.parse(url)`:** RSS-Feed abrufen und dessen XML-Struktur verarbeiten.
- **`feed.entries`:** Liste der einzelnen RSS-Beiträge zurückgeben.
- **`entry.title`:** Artikeltitel auslesen.
- **`entry.link`:** URL des Artikels auslesen.
- **`entry.published`:** Veröffentlichungsdatum als Text auslesen.
- **`entry.published_parsed`:** Veröffentlichungsdatum als strukturierte Zeitangabe auslesen, sofern vorhanden.
- **`entry.summary`:** Zusammenfassung beziehungsweise Beschreibung auslesen.
- **`entry.get('title', '')`:** Feld auslesen und bei fehlendem Feld einen leeren Text zurückgeben.
- **`pd.DataFrame()`:** Extrahierte RSS-Beiträge tabellarisch speichern.

### RSS Feed Aggregation

- **Definition:** Zusammenführen mehrerer RSS-Feeds unterschiedlicher Quellen.
- **RSS Aggregator:** Anwendung, die mehrere RSS-Feeds automatisch abruft und ihre Inhalte gemeinsam bereitstellt.
- **Zweck:** Nachrichten und andere Aktualisierungen verschiedener Webseiten zentral sammeln und auswerten.

### Erkenntnisse

- RSS basiert auf XML und stellt regelmässig aktualisierte Inhalte in einem standardisierten Format bereit.
- `feedparser.parse()` liest den Feed und wandelt seine Inhalte in einfach zugängliche Python-Strukturen um.
- `feed.entries` enthält die einzelnen Beiträge eines Feeds.
- RSS eignet sich für die automatisierte Sammlung von Artikeltiteln, Links, Beschreibungen und Veröffentlichungsdaten.
- **Unterschied zu Web Scraping:** RSS liefert bereits strukturierte Informationen über neue Beiträge. Scraping extrahiert Informationen direkt aus HTML-Webseiten.
- **Achtung:** RSS-Feeds enthalten nicht immer sämtliche Felder und liefern häufig nur einen Teil des vollständigen Artikelinhalts.

In [7]:
# Import libraries
import feedparser                                                               #RSS-Feeds abrufen und XML-Inhalte interpretieren
import pandas as pd                                                             #Extrahierte RSS-Beiträge in einem DataFrame speichern

# Define RSS feed URL
feed_url = 'https://www.nzz.ch/startseite.rss'                                  #RSS-Feed der NZZ als Datenquelle festlegen

# Parse the RSS feed
feed = feedparser.parse(feed_url)                                               #RSS-Feed von der URL abrufen und XML-Inhalte parsen
                                                                                #Ergebnis: Feed mit Metadaten und einzelnen Artikeln

# Extract feed entries
feed_entries = feed.entries                                                     #Liste aller vom Feed bereitgestellten Beiträge auslesen

# Create an empty list
rows = []                                                                       #Leere Liste zum Sammeln der extrahierten Artikelinformationen erstellen

# Extract information from RSS entries
for entry in feed_entries[:10]:                                                 #Die ersten zehn Beiträge des RSS-Feeds durchlaufen
                                                                                #[:10]: Maximal zehn Einträge auswählen

    article_title = entry.get('title', '')                                      #Artikeltitel auslesen; bei fehlendem Feld leeren Text verwenden

    article_link = entry.get('link', '')                                        #URL zum vollständigen Artikel auslesen

    article_published = entry.get('published', '')                              #Veröffentlichungsdatum als Text auslesen

    article_summary = entry.get('summary', '')                                  #Beschreibung beziehungsweise Artikelzusammenfassung auslesen

    rows.append({                                                               #Extrahierte Informationen als Dictionary speichern
        'title': article_title,                                                 #Artikeltitel
        'link': article_link,                                                   #URL des Artikels
        'published': article_published,                                         #Veröffentlichungsdatum
        'summary': article_summary                                              #Artikelzusammenfassung
    })

# Create a DataFrame
df_rss = pd.DataFrame(
    rows,
    columns=['title', 'link', 'published', 'summary']
)                                                                               #RSS-Beiträge in tabellarische Struktur umwandeln
                                                                                #columns: Gewünschte Spaltennamen und Reihenfolge definieren
                                                                                #Auch bei leerem Feed werden die angegebenen Spalten erstellt

# Show results
print('Number of articles:', len(df_rss))                                       #Anzahl extrahierter Artikel anzeigen
df_rss.head()                                                                   #Erste fünf Beiträge anzeigen

# Save RSS data
df_rss.to_csv('rss_articles.csv', index=False)                                  #RSS-Beiträge als CSV-Datei speichern
                                                                                #index=False: Automatischen Zeilenindex nicht mitspeichern

#ANPASSEN: feed_url durch die URL des gewünschten RSS-Feeds ersetzen
#ANPASSEN: [:10] durch die gewünschte Anzahl Beiträge ersetzen; ohne [:10] alle verfügbaren Beiträge verarbeiten
#ANPASSEN: 'title', 'link', 'published' und 'summary' an die im Feed vorhandenen Felder anpassen
#ANPASSEN: 'rss_articles.csv' an den gewünschten Ausgabedateinamen anpassen
#NICHTS ANPASSEN: feedparser.parse(), feed.entries, entry.get(), pd.DataFrame() und .to_csv() als Methoden
#VORAUSSETZUNG: Internetzugang; feedparser und pandas installiert
#ACHTUNG: Fehlende RSS-Felder werden durch entry.get('feld', '') als leere Zeichenketten behandelt
#ACHTUNG: Bei Netzwerkproblemen oder ungültigen Feed-URLs kann feed.entries leer sein
#ERKENNTNIS: RSS-Beiträge werden über eine standardisierte XML-Schnittstelle abgerufen und als DataFrame gespeichert

ModuleNotFoundError: No module named 'feedparser'